# Food101 ViT Baseline - Vision Transformer

## Experiment Design
- **Task:** Single-label food image classification
- **Dataset:** Food-101 (101 classes, 75,750 train / 25,250 validation)
- **Model:** Vision Transformer (ViT-B/16)
- **Framework:** PyTorch Lightning
- **Reproducibility:** Fixed seed = 42
- **Training Budget:** 20 epochs with early stopping
- **Optimizer:** AdamW (lr=1e-4, weight_decay=1e-4)
- **Scheduler:** ReduceLROnPlateau (factor=0.5, patience=2)
- **Augmentation:** RandomResizedCrop, HorizontalFlip, ColorJitter
- **Metrics:** Top-1 Accuracy, Macro-F1

## Section 0: Setup & Dependencies

In [ ]:
# Install dependencies (if needed)
# !pip install -q pytorch-lightning torchmetrics timm datasets onnx onnxruntime

In [ ]:
# Imports
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import torchmetrics
import timm
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import json

In [ ]:
# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Seed fixing function
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set seed for reproducibility
set_seed(42)
print("✅ Seed set to 42")

In [ ]:
# Shared configuration
SHARED_CONFIG = {
    "seed": 42,
    "image_size": 224,
    "batch_size": 64,
    "num_workers": 4,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "max_epochs": 20,
    "patience_early_stop": 5,
    "patience_lr_reduce": 2,
    "precision": "16-mixed",
}

print("Configuration:")
for k, v in SHARED_CONFIG.items():
    print(f"  {k}: {v}")

## Section 1: Dataset Preparation

In [ ]:
# Load Food101 from Hugging Face
print("Loading Food101 dataset from Hugging Face...")
dataset = load_dataset("ethz/food101")
print("✅ Dataset loaded")
print(f"Train split: {len(dataset['train'])} images")
print(f"Validation split: {len(dataset['validation'])} images")

In [ ]:
# Create id2label and label2id mappings
class_names = dataset["train"].features["label"].names
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in id2label.items()}
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Sample classes: {class_names[:10]}")

In [ ]:
# Visualize sample images
sample = dataset["train"].shuffle(seed=42).select(range(9))

plt.figure(figsize=(12, 12))
for i, item in enumerate(sample):
    plt.subplot(3, 3, i + 1)
    plt.imshow(item["image"])
    plt.title(id2label[item["label"]])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Define transforms with enhanced augmentation
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("✅ Transforms defined")

In [ ]:
# HFImageDataset wrapper
class HFImageDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.ds = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"]
        label = item["label"]

        # Ensure 3-channel RGB
        if img.mode != "RGB":
            img = img.convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

print("✅ HFImageDataset defined")

In [ ]:
# Create datasets and dataloaders
train_ds = HFImageDataset(dataset["train"], transform=train_tfms)
val_ds = HFImageDataset(dataset["validation"], transform=val_tfms)

train_dl = DataLoader(
    train_ds,
    batch_size=SHARED_CONFIG["batch_size"],
    shuffle=True,
    num_workers=SHARED_CONFIG["num_workers"],
    pin_memory=True,
)

val_dl = DataLoader(
    val_ds,
    batch_size=SHARED_CONFIG["batch_size"],
    shuffle=False,
    num_workers=SHARED_CONFIG["num_workers"],
    pin_memory=True,
)

print(f"✅ Dataloaders created")
print(f"Train batches: {len(train_dl)}")
print(f"Val batches: {len(val_dl)}")

## Section 2: Model Definition

In [ ]:
# ViT model candidates
VIT_MODELS = {
    "vit_base_patch16_224": {"params": "86.6M", "notes": "Original ViT-B/16, strong baseline"},
    "deit_base_patch16_224": {"params": "86.6M", "notes": "DeiT with better training recipe"},
    "vit_small_patch16_224": {"params": "22.1M", "notes": "Lighter ViT, faster convergence"},
}

# Primary model for this notebook
BACKBONE_NAME = "vit_base_patch16_224"

print(f"Selected model: {BACKBONE_NAME}")
print(f"Model info: {VIT_MODELS[BACKBONE_NAME]}")

In [ ]:
# Enhanced FoodClassifier with torchmetrics
class FoodClassifier(pl.LightningModule):
    def __init__(self, backbone_name: str, num_classes: int, lr: float = 1e-4, seed: int = 42):
        super().__init__()
        self.save_hyperparameters()
        pl.seed_everything(seed)

        self.model = timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()

        # Metrics
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_f1 = torchmetrics.F1Score(
            task="multiclass", num_classes=num_classes, average="macro"
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)

        self.train_acc(preds, y)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)

        self.val_acc(preds, y)
        self.val_f1(preds, y)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_f1", self.val_f1, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr, weight_decay=SHARED_CONFIG["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=SHARED_CONFIG["patience_lr_reduce"]
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

print("✅ FoodClassifier defined")

In [ ]:
# Test model forward pass
test_model = FoodClassifier(BACKBONE_NAME, num_classes, lr=SHARED_CONFIG["lr"], seed=SHARED_CONFIG["seed"])
test_input = torch.randn(2, 3, 224, 224)
test_output = test_model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Expected: (2, {num_classes})")
del test_model, test_input, test_output

## Section 3: Training

In [ ]:
# Initialize model
model = FoodClassifier(
    backbone_name=BACKBONE_NAME,
    num_classes=num_classes,
    lr=SHARED_CONFIG["lr"],
    seed=SHARED_CONFIG["seed"],
)

print(f"✅ Model initialized: {BACKBONE_NAME}")

In [ ]:
# Setup callbacks
checkpoint_cb = ModelCheckpoint(
    dirpath="checkpoints/food101_vit",
    filename=f"{BACKBONE_NAME}-{{epoch:02d}}-{{val_acc:.4f}}-{{val_f1:.4f}}",
    monitor="val_acc",
    mode="max",
    save_top_k=1,
)

early_stop_cb = EarlyStopping(
    monitor="val_acc",
    mode="max",
    patience=SHARED_CONFIG["patience_early_stop"],
)

print("✅ Callbacks configured")

In [ ]:
# Create trainer
trainer = pl.Trainer(
    max_epochs=SHARED_CONFIG["max_epochs"],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
    precision=SHARED_CONFIG["precision"],
    deterministic=True,
)

print("✅ Trainer created")
print(f"Device: {trainer.accelerator}")
print(f"Precision: {SHARED_CONFIG['precision']}")

In [ ]:
# Train model
print("🚀 Starting training...")
print("Note: ViTs may require more epochs (20-30) to converge fully")
trainer.fit(model, train_dl, val_dl)
print("✅ Training complete!")

In [ ]:
# Display best checkpoint info
print(f"Best checkpoint: {checkpoint_cb.best_model_path}")
print(f"Best val_acc: {checkpoint_cb.best_model_score:.4f}")

## Section 4: Evaluation

In [ ]:
# Evaluation function
def evaluate_model(model, dataloader, id2label, device="cuda"):
    model.eval()
    model.to(device)
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Evaluating"):
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    # Classification report
    print("\n" + "="*50)
    print("Classification Report")
    print("="*50)
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[id2label[i] for i in range(len(id2label))],
            digits=4,
        )
    )

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(20, 20))
    sns.heatmap(
        cm,
        annot=False,
        fmt="d",
        cmap="Blues",
        xticklabels=[id2label[i] for i in range(len(id2label))],
        yticklabels=[id2label[i] for i in range(len(id2label))],
    )
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return all_preds, all_labels

print("✅ Evaluation function defined")

In [ ]:
# Load best checkpoint
best_model = FoodClassifier.load_from_checkpoint(
    checkpoint_cb.best_model_path,
    backbone_name=BACKBONE_NAME,
    num_classes=num_classes,
)
best_model.eval()
print(f"✅ Best model loaded from {checkpoint_cb.best_model_path}")

In [ ]:
# Evaluate on validation set
device = "cuda" if torch.cuda.is_available() else "cpu"
all_preds, all_labels = evaluate_model(best_model, val_dl, id2label, device)

In [ ]:
# Per-class performance (top-5 worst classes)
from sklearn.metrics import recall_score

per_class_recall = []
for i in range(num_classes):
    mask = np.array(all_labels) == i
    if mask.sum() > 0:
        recall = (np.array(all_preds)[mask] == i).mean()
        per_class_recall.append((id2label[i], recall))

# Sort by recall (ascending)
per_class_recall.sort(key=lambda x: x[1])

print("\nTop-5 Worst Performing Classes:")
for class_name, recall in per_class_recall[:5]:
    print(f"  {class_name}: {recall:.4f}")

In [ ]:
# Visualize sample predictions
def visualize_predictions(model, dataset, id2label, num_samples=9, device="cuda"):
    model.eval()
    model.to(device)
    
    sample_indices = random.sample(range(len(dataset)), num_samples)
    
    plt.figure(figsize=(15, 15))
    for i, idx in enumerate(sample_indices):
        img_tensor, true_label = dataset[idx]
        
        # Predict
        with torch.no_grad():
            logits = model(img_tensor.unsqueeze(0).to(device))
            pred_label = logits.argmax(dim=1).item()
        
        # Denormalize image for display
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_display = img_tensor * std + mean
        img_display = torch.clamp(img_display, 0, 1)
        img_display = img_display.permute(1, 2, 0).numpy()
        
        # Plot
        plt.subplot(3, 3, i + 1)
        plt.imshow(img_display)
        color = "green" if pred_label == true_label else "red"
        plt.title(
            f"True: {id2label[true_label]}\nPred: {id2label[pred_label]}",
            color=color,
            fontsize=10,
        )
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

visualize_predictions(best_model, val_ds, id2label, num_samples=9, device=device)

## Section 5: Model Export

In [ ]:
# Load best checkpoint for export
export_model = FoodClassifier.load_from_checkpoint(
    checkpoint_cb.best_model_path,
    backbone_name=BACKBONE_NAME,
    num_classes=num_classes,
)
export_model.eval()
export_model.cpu()

print("✅ Model loaded for export")

In [ ]:
# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = f"../food101_{BACKBONE_NAME}.onnx"

torch.onnx.export(
    export_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    input_names=["input"],
    output_names=["logits"],
    training=torch.onnx.TrainingMode.EVAL,
    dynamo=False,
)

print(f"✅ ONNX model saved to {onnx_path}")

In [ ]:
# Save id2label mapping (only if not already created by CNN notebook)
label_path = "../food101_id2label.json"
with open(label_path, "w") as f:
    json.dump(id2label, f, indent=2)

print(f"✅ Labels saved to {label_path}")

In [ ]:
# Verify ONNX export
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)
test_input = torch.randn(1, 3, 224, 224).numpy()
onnx_output = session.run(None, {"input": test_input})[0]

print(f"ONNX input shape: {test_input.shape}")
print(f"ONNX output shape: {onnx_output.shape}")
print(f"✅ ONNX export verified")

## Summary

### Training Results
- **Model:** ViT-B/16 (Vision Transformer)
- **Best Validation Accuracy:** See checkpoint filename
- **Best Validation Macro-F1:** See checkpoint filename
- **Training completed with early stopping**

### Outputs
- **Checkpoint:** `checkpoints/food101_vit/{model}-epoch-val_acc-val_f1.ckpt`
- **ONNX Model:** `food101_vit_base_patch16_224.onnx`
- **Labels:** `food101_id2label.json`

### Notes
- ViTs typically achieve higher accuracy than CNNs but require more epochs
- Expected validation accuracy: 82-87%
- Larger model size (86.6M params) compared to ResNet50 (25.6M params)